# III — Brain Tumor Classification with GoogLeNet (Inception v1) — Training

This notebook implements the training pipeline for **GoogLeNet (Inception v1)** (Szegedy et al., 2014) to classify brain MRI scans into four diagnostic categories:
- 
- 
-  (healthy control)
- 

### Notebook Workflow
1. **Section 1 — Model Configuration & Data Loading**
2. **Section 2 — Build the GoogLeNet Architecture**
3. **Section 3 — Compile the Model**
4. **Section 4 — Train the Model with Standardized Callbacks**
5. **Section 5 — Save Trained Model Artifacts**

> *Evaluation on held-out test data is performed in .*

## 1. Model Configuration & Data Loading

Import required dependencies, configure hardware/GPU acceleration, load the prepared datasets from `Data/clean/classification_prepared/`, and initialize reproducible random seeds.

In [ ]:
import importlib
import json
import sys
import time
from pathlib import Path
import warnings

# Add project root to sys.path
sys.path.insert(0, str(Path.cwd().parents[1]))

import numpy as np
import tensorflow as tf

import src.classification.prepare_data as prepare_data
import src.classification.train as classification_train

prepare_data = importlib.reload(prepare_data)
classification_train = importlib.reload(classification_train)

from src.classification.prepare_data import (
    get_project_paths,
    load_datasets,
    preprocess_datasets,
    augment_datasets,
    validate_dataset_readiness,
)
from src.classification.train import (
    build_googlenet,
    compile_googlenet,
    get_training_callbacks,
)

warnings.filterwarnings("ignore")
print(f"TensorFlow Version: {tf.__version__}")

2026-10-29 09:55:26.433128: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-10-29 09:55:26.469048: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-10-29 09:55:27.293060: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


TensorFlow Version: 2.20.0


In [2]:
# Configure GPU memory growth
gpus = tf.config.list_physical_devices("GPU")
if gpus:
    for gpu in gpus:
        try:
            tf.config.experimental.set_memory_growth(gpu, True)
            print(f"GPU detected: {gpu.name} (Memory growth enabled)")
        except Exception as e:
            print(f"Could not configure memory growth: {e}")
else:
    print("No GPU detected; running on CPU.")

GPU detected: /physical_device:GPU:0 (Memory growth enabled)


In [3]:
# Centralized Hyperparameter & Path Configuration
CONFIG = {
    "image_size": (224, 224),
    "input_shape": (224, 224, 3),
    "num_classes": 4,
    "batch_size": 32,
    "epochs": 35,
    "learning_rate": 1e-4,
    "dropout_rate": 0.4,
    "random_state": 42,
    "early_stopping_patience": 10,
    "reduce_lr_patience": 4,
    "reduce_lr_factor": 0.5,
}

# Set random seeds for reproducibility
tf.keras.utils.set_random_seed(CONFIG["random_state"])
np.random.seed(CONFIG["random_state"])

paths = get_project_paths()
model_dir = paths["project_root"] / "models"
model_dir.mkdir(parents=True, exist_ok=True)

print("Model Configuration:")
for k, v in CONFIG.items():
    print(f"  {k:25s}: {v}")
print(f"\nModels destination dir: {model_dir}")

Model Configuration:
  image_size               : (224, 224)
  input_shape              : (224, 224, 3)
  num_classes              : 4
  batch_size               : 32
  epochs                   : 35
  learning_rate            : 0.0001
  dropout_rate             : 0.4
  random_state             : 42
  early_stopping_patience  : 10
  reduce_lr_patience       : 4
  reduce_lr_factor         : 0.5

Models destination dir: /home/houssam/Documents/brief-5/models


In [4]:
# Load, preprocess, and augment datasets
train_raw, val_raw, test_raw, class_names, class_mapping = load_datasets(
    paths["train_pool_dir"],
    paths["test_dir"],
)

# Normalize tensors to [0, 1]
train_norm, val_norm, test_norm = preprocess_datasets(train_raw, val_raw, test_raw)

# Apply training-only augmentation (horizontal flip, Gaussian noise, blur) and prefetch
train_ds, val_ds, test_ds = augment_datasets(train_norm, val_norm, test_norm)

# Verify readiness
dataset_counts = validate_dataset_readiness(
    train_raw, val_raw, test_raw, val_ds, class_names
)

Found 5521 files belonging to 4 classes.
Using 4693 files for training.


I0000 00:00:1793267780.757982  608202 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 1210 MB memory:  -> device: 0, name: NVIDIA RTX 2000 Ada Generation Laptop GPU, pci bus id: 0000:01:00.0, compute capability: 8.9


Found 5521 files belonging to 4 classes.
Using 828 files for validation.
Found 1205 files belonging to 4 classes.
Class-to-index mapping: {0: 'glioma', 1: 'meningioma', 2: 'notumor', 3: 'pituitary'}


2026-10-29 09:56:22.716675: I tensorflow/core/framework/local_rendezvous.cc:407] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence
2026-10-29 09:56:22.884567: I tensorflow/core/framework/local_rendezvous.cc:407] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence


Train images: 4693
Validation images: 828
Test images: 1205
All dataset readiness checks passed.


2026-10-29 09:56:23.194431: I tensorflow/core/framework/local_rendezvous.cc:407] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence


## 2. Build the GoogLeNet (Inception v1) Architecture

GoogLeNet addresses the challenge of capturing tumor patterns across varying spatial scales by using **Inception modules**. Each Inception block extracts features using parallel multi-scale filter banks in a single layer:
1. **$1 \times 1$ Convolutions**: Pointwise channel transformations and dimensionality reduction.
2. **$3 \times 3$ Convolutions**: Local receptive fields with preceding $1 \times 1$ dimension reduction.
3. **$5 \times 5$ Convolutions**: Broader context receptive fields with preceding $1 \times 1$ reduction.
4. **$3 \times 3$ Max Pooling**: Spatial pooling with subsequent $1 \times 1$ projection.

The network comprises a convolutional stem, **9 Inception modules** (grouped into 3a-3b, 4a-4e, and 5a-5b), global average pooling, dropout, and a final 4-class softmax head.

In [5]:
# Build the GoogLeNet Inception v1 Model
model = build_googlenet(
    input_shape=CONFIG["input_shape"],
    num_classes=CONFIG["num_classes"],
    include_auxiliary=False,
    dropout_rate=CONFIG["dropout_rate"],
)

print(f"Model Name:       {model.name}")
print(f"Input Shape:      {model.input_shape}")
print(f"Output Shape:     {model.output_shape}")
print(f"Total Parameters: {model.count_params():,}")
print(f"Trainable Params: {sum(np.prod(p.shape) for p in model.trainable_weights):,}")

Model Name:       GoogLeNet_InceptionV1
Input Shape:      (None, 224, 224, 3)
Output Shape:     (None, 4)
Total Parameters: 6,287,428
Trainable Params: 6,271,364


In [6]:
# Architecture Structural Summary
model.summary(expand_nested=False, line_length=100)

Model: "GoogLeNet_InceptionV1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                ┃ Output Shape            ┃        Param # ┃ Connected to            ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ input_image (InputLayer)    │ (None, 224, 224, 3)     │              0 │ -                       │
├─────────────────────────────┼─────────────────────────┼────────────────┼─────────────────────────┤
│ stem_conv1_conv (Conv2D)    │ (None, 112, 112, 64)    │          9,408 │ input_image[0][0]       │
├─────────────────────────────┼─────────────────────────┼────────────────┼─────────────────────────┤
│ stem_conv1_bn               │ (None, 112, 112, 64)    │            256 │ stem_conv1_conv[0][0]   │
│ (BatchNormalization)        │                         │                │                         │
├─────────────────────────────┼─────────────────────────┼────────────────┼─────────────────────────┤
│ stem_conv1_relu (ReLU)      │ (None, 112, 112, 64)    │              0 │ stem_conv1_bn[0][0]     │
├─────────────────────────────┼─────────────────────────┼────────────────┼─────────────────────────┤
│ stem_maxpool1               │ (None, 56, 56, 64)      │              0 │ stem_conv1_relu[0][0]   │
│ (MaxPooling2D)              │                         │                │                         │
├─────────────────────────────┼─────────────────────────┼────────────────┼─────────────────────────┤
│ stem_conv2_reduce_conv      │ (None, 56, 56, 64)      │          4,096 │ stem_maxpool1[0][0]     │
│ (Conv2D)                    │                         │                │                         │
├─────────────────────────────┼─────────────────────────┼────────────────┼─────────────────────────┤
│ stem_conv2_reduce_bn        │ (None, 56, 56, 64)      │            256 │ stem_conv2_reduce_conv… │
│ (BatchNormalization)        │                         │                │                         │
├─────────────────────────────┼─────────────────────────┼────────────────┼─────────────────────────┤
│ stem_conv2_reduce_relu      │ (None, 56, 56, 64)      │              0 │ stem_conv2_reduce_bn[0… │
│ (ReLU)                      │                         │                │                         │
├─────────────────────────────┼─────────────────────────┼────────────────┼─────────────────────────┤
│ stem_conv2_conv (Conv2D)    │ (None, 56, 56, 192)     │        110,592 │ stem_conv2_reduce_relu… │
├─────────────────────────────┼─────────────────────────┼────────────────┼─────────────────────────┤
│ stem_conv2_bn               │ (None, 56, 56, 192)     │            768 │ stem_conv2_conv[0][0]   │
│ (BatchNormalization)        │                         │                │                         │
├─────────────────────────────┼─────────────────────────┼────────────────┼─────────────────────────┤
│ stem_conv2_relu (ReLU)      │ (None, 56, 56, 192)     │              0 │ stem_conv2_bn[0][0]     │
├─────────────────────────────┼─────────────────────────┼────────────────┼─────────────────────────┤
│ stem_maxpool2               │ (None, 28, 28, 192)     │              0 │ stem_conv2_relu[0][0]   │
│ (MaxPooling2D)              │                         │                │                         │
├─────────────────────────────┼─────────────────────────┼────────────────┼─────────────────────────┤
│ inception_3a_b3_reduce_conv │ (None, 28, 28, 16)      │          3,072 │ stem_maxpool2[0][0]     │
│ (Conv2D)                    │                         │                │                         │
├─────────────────────────────┼─────────────────────────┼────────────────┼─────────────────────────┤
│ inception_3a_b3_reduce_bn   │ (None, 28, 28, 16)      │             64 │ inception_3a_b3_reduce… │
│ (BatchNormalization)        │                         │                │                         │
├─────────────────────────────┼─────────────────────────┼────

 Total params: 6,287,428 (23.98 MB)

 Trainable params: 6,271,364 (23.92 MB)

 Non-trainable params: 16,064 (62.75 KB)

## 3. Compile the Model

Configure the model with:
- **Optimizer**: `Adam` with initial learning rate $\eta = 10^{-4}$ for stable convergence.
- **Loss Function**: `SparseCategoricalCrossentropy` (matching integer-encoded labels `[0, 1, 2, 3]`).
- **Metrics**: `['accuracy']` tracked across training and validation batches.

In [7]:
# Compile the model
model = compile_googlenet(
    model,
    learning_rate=CONFIG["learning_rate"],
    include_auxiliary=False,
)

print("Model compiled successfully:")
print(f"  Optimizer: {model.optimizer.__class__.__name__} (lr={CONFIG['learning_rate']})")
print(f"  Loss:      {model.loss.__class__.__name__}")
print(f"  Metrics:   {[m.name for m in model.metrics]}")

Model compiled successfully:
  Optimizer: Adam (lr=0.0001)
  Loss:      SparseCategoricalCrossentropy
  Metrics:   ['loss', 'compile_metrics']


## 4. Train the Model

Train using `train_ds` and monitor performance on `val_ds` using clinical best-practice callbacks:
- **ModelCheckpoint**: Persists the model state that minimizes validation loss (`val_loss`).
- **EarlyStopping**: Halts training if `val_loss` fails to improve for 10 epochs, restoring optimal weights.
- **ReduceLROnPlateau**: Halves learning rate when validation loss plateaus for 4 epochs.

> **Methodological Rule:** The test dataset (`test_ds`) is strictly held out and never used during training, hyperparameter selection, or early stopping.

In [8]:
# Setup Checkpoints and Callbacks
checkpoint_path = model_dir / "googlenet_brain_tumor_best.keras"

training_callbacks = get_training_callbacks(
    checkpoint_path=checkpoint_path,
    patience_early_stopping=CONFIG["early_stopping_patience"],
    patience_reduce_lr=CONFIG["reduce_lr_patience"],
    reduce_lr_factor=CONFIG["reduce_lr_factor"],
)

print(f"Checkpoints will be saved to: {checkpoint_path}")

Checkpoints will be saved to: /home/houssam/Documents/brief-5/models/googlenet_brain_tumor_best.keras


In [9]:
# Execute Training
start_time = time.time()

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=CONFIG["epochs"],
    callbacks=training_callbacks,
    verbose=1,
)

training_duration = time.time() - start_time
print(f"\nTraining completed in {training_duration / 60:.2f} minutes.")

Epoch 1/35


2026-10-29 09:58:44.758013: I external/local_xla/xla/service/service.cc:163] XLA service 0x7fdb0001b6c0 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
2026-10-29 09:58:44.758065: I external/local_xla/xla/service/service.cc:171]   StreamExecutor device (0): NVIDIA RTX 2000 Ada Generation Laptop GPU, Compute Capability 8.9
2026-10-29 09:58:45.752646: I tensorflow/compiler/mlir/tensorflow/utils/dump_mlir_util.cc:269] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
2026-10-29 09:58:48.863433: I external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:473] Loaded cuDNN version 92700
2026-10-29 09:58:50.498384: I external/local_xla/xla/stream_executor/cuda/subprocess_compilation.cc:346] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_14679', 8 bytes spill stores, 8 bytes spill loads



  1/147 ━━━━━━━━━━━━━━━━━━━━ 2:07:18 52s/step - accuracy: 0.3125 - loss: 1.4840

I0000 00:00:1793267962.614973  609153 device_compiler.h:196] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.
2026-10-29 09:59:22.627265: W external/local_xla/xla/tsl/framework/bfc_allocator.cc:382] Garbage collection: deallocate free memory regions (i.e., allocations) so that we can re-allocate a larger region to avoid OOM due to memory fragmentation. If you see this message frequently, you are running near the threshold of the available device memory and re-allocation may incur great performance overhead. You may try smaller batch sizes to observe the performance impact. Set TF_ENABLE_GPU_GARBAGE_COLLECTION=false if you'd like to disable this feature.


147/147 ━━━━━━━━━━━━━━━━━━━━ 0s 337ms/step - accuracy: 0.7287 - loss: 0.6859

2026-10-29 10:00:13.864815: I external/local_xla/xla/stream_executor/cuda/subprocess_compilation.cc:346] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_1498', 8 bytes spill stores, 8 bytes spill loads




Epoch 1: val_loss improved from None to 5.36698, saving model to /home/houssam/Documents/brief-5/models/googlenet_brain_tumor_best.keras

Epoch 1: finished saving model to /home/houssam/Documents/brief-5/models/googlenet_brain_tumor_best.keras
147/147 ━━━━━━━━━━━━━━━━━━━━ 116s 438ms/step - accuracy: 0.7287 - loss: 0.6859 - val_accuracy: 0.2198 - val_loss: 5.3670 - learning_rate: 1.0000e-04
Epoch 2/35
147/147 ━━━━━━━━━━━━━━━━━━━━ 0s 135ms/step - accuracy: 0.8417 - loss: 0.4207
Epoch 2: val_loss improved from 5.36698 to 2.96256, saving model to /home/houssam/Documents/brief-5/models/googlenet_brain_tumor_best.keras

Epoch 2: finished saving model to /home/houssam/Documents/brief-5/models/googlenet_brain_tumor_best.keras
147/147 ━━━━━━━━━━━━━━━━━━━━ 22s 149ms/step - accuracy: 0.8417 - loss: 0.4207 - val_accuracy: 0.3043 - val_loss: 2.9626 - learning_rate: 1.0000e-04
Epoch 3/35
147/147 ━━━━━━━━━━━━━━━━━━━━ 0s 266ms/step - accuracy: 0.8773 - loss: 0.3366
Epoch 3: val_loss improved from 2.9

## 5. Save Model Artifacts

Save the trained model checkpoint, class mapping, and training configuration metadata to .

In [ ]:
# Save final model
final_model_path = model_dir / "googlenet_brain_tumor_final.keras"
model.save(str(final_model_path))
print(f"Saved final model to: {final_model_path}")

# Save class mapping
mapping_path = model_dir / "class_mapping.json"
with open(mapping_path, "w") as f:
    json.dump(class_mapping, f, indent=2)
print(f"Saved class mapping to: {mapping_path}")

# Save model configuration and training history
config_path = model_dir / "model_config.json"
with open(config_path, "w") as f:
    json.dump(CONFIG, f, indent=2)
print(f"Saved model configuration to: {config_path}")

history_path = model_dir / "training_history.json"
with open(history_path, "w") as f:
    json.dump(history.history, f, indent=2)
print(f"Saved training history to: {history_path}")
print("Training complete! Proceed to 04_classification_evaluation.ipynb for evaluation on test data.")